# 02  STM youtube_sent (nivel sentenca, pipeline independente)

Config e treinamento separados de 01_stm_youtube_doc.ipynb. Ver
docs/superpowers/specs/2026-09-20-stm-youtube-sent-design.md.

## 1. Setup e imports

In [1]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
# Cell 1 — Imports
import sys, os, json, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from _helpers import load_params, get_corpus_config, get_seed, load_corpus
from _helpers import lemmatize_corpus
from _helpers import (
    prepare_stm_input, grid_search_k_stm, grid_search_stm_hparams, train_stm,
    qualitative_report,
)
from _helpers import name_all_topics, name_topic
from _helpers import (
    compute_coherence_cv, compute_exclusivity, compute_exclusivity_ctfidf, compute_diversity,
    compute_topic_diversity, compute_frex_score, compute_stability,
    export_results, export_topics_for_eval,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100


D:\Área de Trabalho\UPE\1 Período\PLN\Topic Modeling - LDA and BERTopic\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2  Carrega params do params.yaml
CORPUS_ID = "youtube_sent"
LANG = "en"

params = load_params()
_, cfg = get_corpus_config(params, CORPUS_ID)
seed = get_seed(params)
TEXT_COL = cfg["text_column"]
ID_COL = cfg["post_id_column"]
PREV_FORMULA = cfg.get("stm_prevalence_formula", "") or ""
MIN_TOKENS = int(cfg.get("stm_min_tokens_per_doc", 0) or 0)
assert PREV_FORMULA, "stm_prevalence_formula vazia no params.yaml  defina a formula do corpus"

print(f"corpus_id = {CORPUS_ID}")
print(f"text_column = {TEXT_COL}")
print(f"post_id_column = {ID_COL}")
print(f"language = {LANG}")
print(f"seed = {seed}")
print(f"covariates = {cfg.get('covariates', [])}")
print(f"prevalence_formula = {PREV_FORMULA}")
print(f"min_tokens_per_doc = {MIN_TOKENS}")

corpus_id = youtube_sent
text_column = sentence
post_id_column = sent_id
language = en
seed = 42
covariates = ['product_category']
prevalence_formula = ~ product_category
min_tokens_per_doc = 5


## 2. Corpus de sentencas

In [3]:
# Cell 3  Carrega corpus_sentencas.csv (run mais recente de 02-sentences)
from _helpers import resolve_latest_dir

_corpus_dir = resolve_latest_dir(
    "../../02-sentences/data/output/youtube_sent",
    contains="corpus_sentencas.csv",
)
CORPUS_VERSION = _corpus_dir.name
print(f"Versao do corpus: {CORPUS_VERSION}")

df = pd.read_csv(_corpus_dir / "corpus_sentencas.csv", encoding="utf-8")
print(f"Total de sentencas carregadas: {len(df):,}")
print(f"Colunas: {list(df.columns)}")
df.head(3)

  versão resolvida: youtube_sent_20260920_141442/ (latest)
Versao do corpus: youtube_sent_20260920_141442


Total de sentencas carregadas: 130,016
Colunas: ['sent_id', 'post_id', 'sent_idx', 'sentence', 'context', 'message_raw', 'message_punctuated', 'punctuation_restored', 'n_words', 'is_fragment', 'is_dup_exact', 'upload_date', 'year', 'channel', 'source', 'product_category']


,sent_id,post_id,sent_idx,sentence,context,message_raw,message_punctuated,punctuation_restored,n_words,is_fragment,is_dup_exact,upload_date,year,channel,source,product_category
0,WmhKCq0vfjE_0,WmhKCq0vfjE,0,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,False,52,False,False,2016-02-17,2016,Marques Brownlee,whisper,headphone
1,WmhKCq0vfjE_1,WmhKCq0vfjE,1,I promise I didn't pick them up just because I...,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,False,17,False,False,2016-02-17,2016,Marques Brownlee,whisper,headphone
2,WmhKCq0vfjE_2,WmhKCq0vfjE,2,I picked them up because they already kind of ...,I promise I didn't pick them up just because I...,Hey what's up guys MkBHD here and every once i...,Hey what's up guys MkBHD here and every once i...,False,13,False,False,2016-02-17,2016,Marques Brownlee,whisper,headphone


## 3. Limpeza pre-STM (nao sobrescreve corpus_sentencas.csv)

In [4]:
# Cell 4  Filtros pre-STM: fragmento, duplicata exata, covariavel NA, tamanho minimo
_n0 = len(df)
df = df[(~df["is_fragment"]) & (~df["is_dup_exact"])].reset_index(drop=True)
print(f"Removidas por fragmento/duplicata: {_n0 - len(df)} (restam {len(df)})")

_cov_cols = cfg.get("covariates", [])
_n0 = len(df)
df = df.dropna(subset=_cov_cols).reset_index(drop=True)
print(f"Removidas por covariavel NA ({_cov_cols}): {_n0 - len(df)} (restam {len(df)})")

_n0 = len(df)
df = df[df["n_words"] >= MIN_TOKENS].reset_index(drop=True)
print(f"Removidas por < {MIN_TOKENS} palavras: {_n0 - len(df)} (restam {len(df)})")

docs = df[TEXT_COL].astype(str).tolist()
post_ids = df[ID_COL].astype(str).tolist()
print()
print(f"Total sentencas apos limpeza: {len(docs):,}")
df["product_category"].value_counts()

Removidas por fragmento/duplicata: 13905 (restam 116111)
Removidas por covariavel NA (['product_category']): 0 (restam 116111)
Removidas por < 5 palavras: 0 (restam 116111)

Total sentencas apos limpeza: 116,111


product_category
laptop       65666
phone        30911
other         5535
tablet        3395
headphone     3249
watch         2315
desktop       1228
camera        1221
console       1189
monitor        845
vr             557
Name: count, dtype: int64

## 4. Config local do STM e lematizacao

In [5]:
# Cell 5  STM_CFG local: overrides de no_below/no_above SO deste corpus,
# sem tocar em params["stm"] (que segue valendo para youtube_doc)
STM_CFG = {
    **params["stm"],
    "no_below": cfg["stm_no_below"],
    "no_above": cfg["stm_no_above"],
}
print(f"STM_CFG: no_below={STM_CFG['no_below']} no_above={STM_CFG['no_above']} "
      f"max_em_its={STM_CFG.get('max_em_its')} timeout_sec={STM_CFG.get('timeout_sec')}")

print("Lematizando (single-process spaCy en_core_web_sm)...")
t0 = time.time()
tokenized, dictionary = lemmatize_corpus(
    docs, LANG, {"stm": STM_CFG}, model_key="stm",
    extra_stopwords=cfg.get("stopwords_emojis"),
)
print(f"  done em {time.time()-t0:.0f}s")
print(f"  vocab final: {len(dictionary):,} palavras (apos filter_extremes no_below={STM_CFG['no_below']})")

n_tokens_lemma = [len(t) for t in tokenized]
print(f"  avg tokens/sentenca apos lemmatize: {np.mean(n_tokens_lemma):.1f}")

STM_CFG: no_below=20 no_above=0.5 max_em_its=300 timeout_sec=21600
Lematizando (single-process spaCy en_core_web_sm)...


  done em 202s
  vocab final: 2,760 palavras (apos filter_extremes no_below=20)
  avg tokens/sentenca apos lemmatize: 6.5
